# AR Teacher-Forced Training

This notebook trains `ProgressiveBExampleLM` with pure autoregressive teacher forcing.

- Objective: next-token prediction at every position
- Training path: `teacher_forcing=True`
- B path: active during training
- No MLM or suffix corruption mixed in
- Includes environment setup, progress bar, learning curves, checkpoint save, and checkpoint load


In [ ]:
from __future__ import annotations

from dataclasses import asdict
from pathlib import Path
import platform
import sys
import time

import torch

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src").exists() and (path / "scripts").exists()
)
sys.path.insert(0, str(project_root / "src"))
sys.path.insert(0, str(project_root / "scripts"))

from jakal_net import describe_device, resolve_device
from progressive_b_example import (
    ProgressiveBExampleLM,
    build_char_vocab,
    build_progressive_b_stage_specs,
    compute_next_token_loss,
    estimate_next_token_loss,
    generate_next_tokens,
    perplexity_from_loss,
    sample_next_token_batch,
    split_train_val,
)


In [ ]:
TEXT_FILE = ""
DEFAULT_TEXT = (
    "Jakal-Net explores propagation and transition as separate operators.\n"
    "Progressive B activation starts with stable sequence structure in S.\n"
    "Then the B path grows from light compression to stronger hierarchical bottlenecks.\n"
    "The final readout returns to S so token-level prediction remains anchored.\n"
) * 32

DEVICE = "cpu"
SEED = 0

SEQ_LEN = 48
DIM = 64
BATCH_SIZE = 8
STEPS = 200
EVAL_INTERVAL = 25
EVAL_STEPS = 8
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-2
GRAD_CLIP = 1.0

WARMUP_LAYERS = 2
FINAL_REFINE_LAYERS = 2
LITE_LAYERS = 2
MID_LAYERS = 2
FULL_LAYERS = 1
ROUTE_TOPK = 4

# Leave this as None for full teacher forcing.
# Set an integer like 8 or 16 only if you need lower memory.
TEACHER_FORCING_CHUNK_SIZE: int | None = None

MAX_NEW_TOKENS = 120
PROMPT_TEXT = "Jakal-Net explores "

RUN_NAME = "ar_teacher_forced_demo"
OUTPUT_DIR = "artifacts/notebook_runs"
CHECKPOINT_FILE = f"{RUN_NAME}.pt"


In [ ]:
torch.manual_seed(SEED)
device = resolve_device(DEVICE)

output_dir = (project_root / OUTPUT_DIR).resolve()
output_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = output_dir / CHECKPOINT_FILE

print({
    "project_root": str(project_root),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "torch": torch.__version__,
    "device": describe_device(DEVICE),
    "output_dir": str(output_dir),
    "checkpoint_path": str(checkpoint_path),
    "tqdm_available": tqdm is not None,
    "matplotlib_available": plt is not None,
})


In [ ]:
if TEXT_FILE.strip():
    text_path = Path(TEXT_FILE)
    if not text_path.is_absolute():
        text_path = (project_root / text_path).resolve()
    text = text_path.read_text(encoding="utf-8")
    if not text.strip():
        raise ValueError("TEXT_FILE must not be empty.")
else:
    text_path = None
    text = DEFAULT_TEXT

vocab = build_char_vocab(text)
tokens = vocab.encode(text)
train_tokens, val_tokens = split_train_val(tokens, train_fraction=0.9)

print({
    "text_file": None if text_path is None else str(text_path),
    "vocab_size": vocab.size,
    "total_tokens": int(tokens.numel()),
    "train_tokens": int(train_tokens.numel()),
    "val_tokens": int(val_tokens.numel()),
})

if tokens.numel() <= SEQ_LEN + 1:
    raise ValueError("The corpus must be longer than seq_len + 1.")


In [ ]:
stage_specs = build_progressive_b_stage_specs(
    seq_nodes=SEQ_LEN,
    lite_layers=LITE_LAYERS,
    mid_layers=MID_LAYERS,
    full_layers=FULL_LAYERS,
)

model = ProgressiveBExampleLM(
    vocab_size=vocab.size,
    dim=DIM,
    seq_nodes=SEQ_LEN,
    warmup_layers=WARMUP_LAYERS,
    stage_specs=stage_specs,
    final_refine_layers=FINAL_REFINE_LAYERS,
    route_topk=ROUTE_TOPK,
    implementation="streaming",
)

parameter_count = sum(parameter.numel() for parameter in model.parameters())
print({
    "parameters": parameter_count,
    "stage_specs": [asdict(spec) for spec in stage_specs],
})


In [ ]:
def run_teacher_forced_training(
    model: torch.nn.Module,
    train_tokens: torch.Tensor,
    val_tokens: torch.Tensor,
    *,
    seq_len: int,
    batch_size: int,
    device: torch.device | str,
    steps: int,
    eval_interval: int,
    eval_steps: int,
    learning_rate: float,
    weight_decay: float,
    grad_clip: float | None,
    teacher_forcing_chunk_size: int | None,
) -> dict[str, list[float] | list[int]]:
    model.to(device)
    model.train()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    history: dict[str, list[float] | list[int]] = {
        "train_step_losses": [],
        "eval_steps": [],
        "train_losses": [],
        "val_losses": [],
    }

    iterator = range(1, steps + 1)
    progress = tqdm(iterator, total=steps, desc="teacher-forced AR") if tqdm is not None else iterator
    start_time = time.time()

    for step in progress:
        batch = sample_next_token_batch(
            train_tokens,
            seq_len=seq_len,
            batch_size=batch_size,
            device=device,
            teacher_forcing=True,
        )
        optimizer.zero_grad(set_to_none=True)
        loss, _ = compute_next_token_loss(
            model,
            batch,
            teacher_forcing=True,
            teacher_forcing_chunk_size=teacher_forcing_chunk_size,
        )
        loss.backward()
        if grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
        optimizer.step()

        step_loss = float(loss.item())
        history["train_step_losses"].append(step_loss)

        if tqdm is not None:
            progress.set_postfix(loss=f"{step_loss:.4f}")
        elif step == 1 or step % 10 == 0 or step == steps:
            print(f"step={step:4d}/{steps} loss={step_loss:.4f}")

        if step == 1 or step % eval_interval == 0 or step == steps:
            train_eval = estimate_next_token_loss(
                model,
                train_tokens,
                seq_len=seq_len,
                batch_size=batch_size,
                device=device,
                eval_steps=eval_steps,
                teacher_forcing=True,
                teacher_forcing_chunk_size=teacher_forcing_chunk_size,
            )
            val_eval = estimate_next_token_loss(
                model,
                val_tokens,
                seq_len=seq_len,
                batch_size=batch_size,
                device=device,
                eval_steps=eval_steps,
                teacher_forcing=True,
                teacher_forcing_chunk_size=teacher_forcing_chunk_size,
            )
            history["eval_steps"].append(step)
            history["train_losses"].append(train_eval)
            history["val_losses"].append(val_eval)
            print(
                f"eval step={step:4d} | train_loss={train_eval:.4f} ppl={perplexity_from_loss(train_eval):8.2f} "
                f"| val_loss={val_eval:.4f} ppl={perplexity_from_loss(val_eval):8.2f}"
            )

    elapsed = time.time() - start_time
    print({"elapsed_seconds": round(elapsed, 2), "steps": steps})
    return history


In [ ]:
history = run_teacher_forced_training(
    model,
    train_tokens,
    val_tokens,
    seq_len=SEQ_LEN,
    batch_size=BATCH_SIZE,
    device=device,
    steps=STEPS,
    eval_interval=EVAL_INTERVAL,
    eval_steps=EVAL_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    grad_clip=GRAD_CLIP,
    teacher_forcing_chunk_size=TEACHER_FORCING_CHUNK_SIZE,
)


In [ ]:
if plt is None:
    print("matplotlib is not installed. Skipping curve plot.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["train_step_losses"], label="train step loss")
    axes[0].set_title("Step Loss")
    axes[0].set_xlabel("step")
    axes[0].set_ylabel("loss")
    axes[0].grid(True, alpha=0.25)

    axes[1].plot(history["eval_steps"], history["train_losses"], marker="o", label="train eval")
    axes[1].plot(history["eval_steps"], history["val_losses"], marker="o", label="val eval")
    axes[1].set_title("Eval Loss")
    axes[1].set_xlabel("step")
    axes[1].set_ylabel("loss")
    axes[1].grid(True, alpha=0.25)
    axes[1].legend()

    plt.tight_layout()
    plt.show()


In [ ]:
batch = sample_next_token_batch(
    val_tokens,
    seq_len=SEQ_LEN,
    batch_size=2,
    device=device,
    teacher_forcing=True,
)
loss, logits = compute_next_token_loss(
    model,
    batch,
    teacher_forcing=True,
    teacher_forcing_chunk_size=TEACHER_FORCING_CHUNK_SIZE,
)

print({
    "teacher_forced_loss": float(loss.item()),
    "teacher_forced_ppl": perplexity_from_loss(float(loss.item())),
    "logits_shape": tuple(logits.shape),
    "target_shape": tuple(batch.target.shape),
})


In [ ]:
checkpoint = {
    "model_state_dict": model.state_dict(),
    "config": {
        "vocab_size": vocab.size,
        "dim": DIM,
        "seq_nodes": SEQ_LEN,
        "warmup_layers": WARMUP_LAYERS,
        "final_refine_layers": FINAL_REFINE_LAYERS,
        "route_topk": ROUTE_TOPK,
        "implementation": "streaming",
        "stage_specs": [asdict(spec) for spec in stage_specs],
    },
    "history": history,
    "text_file": None if text_path is None else str(text_path),
    "seed": SEED,
}
torch.save(checkpoint, checkpoint_path)
print({"saved_checkpoint": str(checkpoint_path), "bytes": checkpoint_path.stat().st_size})


In [ ]:
loaded_checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
loaded_stage_specs = [
    type(stage_specs[0])(**spec_dict)
    for spec_dict in loaded_checkpoint["config"]["stage_specs"]
]
loaded_model = ProgressiveBExampleLM(
    vocab_size=loaded_checkpoint["config"]["vocab_size"],
    dim=loaded_checkpoint["config"]["dim"],
    seq_nodes=loaded_checkpoint["config"]["seq_nodes"],
    warmup_layers=loaded_checkpoint["config"]["warmup_layers"],
    stage_specs=loaded_stage_specs,
    final_refine_layers=loaded_checkpoint["config"]["final_refine_layers"],
    route_topk=loaded_checkpoint["config"]["route_topk"],
    implementation=loaded_checkpoint["config"]["implementation"],
)
loaded_model.load_state_dict(loaded_checkpoint["model_state_dict"])
loaded_model.to(device)
loaded_model.eval()

print({
    "loaded_checkpoint": str(checkpoint_path),
    "loaded_eval_points": len(loaded_checkpoint["history"]["eval_steps"]),
})


In [ ]:
prompt = PROMPT_TEXT if PROMPT_TEXT else text[:SEQ_LEN]
prompt_ids = vocab.encode(prompt)
if prompt_ids.numel() == 0:
    raise ValueError("PROMPT_TEXT must encode to at least one token.")

generated_ids = generate_next_tokens(
    loaded_model,
    prompt_ids,
    max_new_tokens=MAX_NEW_TOKENS,
    seq_len=SEQ_LEN,
    device=device,
)

print("prompt:")
print(prompt)
print()
print("generated:")
print(vocab.decode(generated_ids.tolist()))
